In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent        # notebooks/ -> project root
sys.path.insert(0, str(ROOT))

In [2]:
from src.rag.ingestor import Ingestor
from src.rag.embedder import Embedder
from src.rag.retriever import Retriever
from src.rag.vector_db import VectorStore

c:\Users\Lenovo\Desktop\Rag_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def evidence_for(answers):
    """Text evidence paragraphs for one question (tables and figures excluded)."""
    evidence = set()
    for answer in answers["answer"]:
        for e in answer["evidence"]:
            if not e.startswith("FLOAT SELECTED"):
                evidence.add(e)
    return evidence


def run_retrieval(dataset, num_papers, ret, reranker=None,
                  methods=("dense", "keyword", "hybrid", "hybrid+rerank"), k=20):
    records = {method: [] for method in methods}

    for paper in dataset.select(range(num_papers)):
        qas = paper["qas"]
        for question, answers in zip(qas["question"], qas["answers"]):
            evidence = evidence_for(answers)
            if not evidence:
                continue

            cache = {}  # reuse the hybrid results for "hybrid" and "hybrid+rerank"

            for method in methods:
                if method == "hybrid+rerank":
                    if "hybrid" not in cache:
                        cache["hybrid"] = ret.search(question, k=k, paper_id=paper["id"], method="hybrid")
                    chunks = reranker.rerank(question, cache["hybrid"], top_k=k)  # reorder all k, don't drop any
                else:
                    chunks = ret.search(question, k=k, paper_id=paper["id"], method=method)
                    if method == "hybrid":
                        cache["hybrid"] = chunks

                correct_ranks = [
                    rank for rank, chunk in enumerate(chunks, start=1)
                    if chunk["text"] in evidence
                ]
                records[method].append({
                    "correct_ranks": correct_ranks,
                    "n_evidence": len(evidence),
                })

    return records


In [4]:
def compute_metrics(question_records, ks=(1, 5, 10, 20)):
    """Metrics at several cutoffs, from records retrieved with k = max(ks)."""
    n = len(question_records)
    results = {"questions": n}

    for k in ks:
        hit = mrr = recall = precision = 0.0
        for r in question_records:
            ranks = [rank for rank in r["correct_ranks"] if rank <= k]  # only what's in the top k
            if ranks:
                hit += 1
                mrr += 1 / ranks[0]
            recall += len(ranks) / r["n_evidence"]
            precision += len(ranks) / k

        results[f"hit@{k}"] = hit / n
        results[f"MRR@{k}"] = mrr / n
        results[f"recall@{k}"] = recall / n
        results[f"precision@{k}"] = precision / n

    return results

In [5]:

embedder = Embedder()

ingestor = Ingestor(str(ROOT / "data" / "qasper_validation.parquet"))
vectorstore = VectorStore(db_path = str(ROOT / "data" / "rag.db"))


ret = Retriever(embedder, vectorstore)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1087.34it/s]
c:\Users\Lenovo\Desktop\Rag_Project\src\rag\embedder.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dimension = self.model.get_sentence_embedding_dimension()


In [10]:
# ingestor.ingest(embedder, vectorstore) 
print(vectorstore.count())

dataset = ingestor.load_qasper_validation()

for paper in dataset:
    paper_id = paper["id"]
    title = paper["title"]
    abstract = paper["abstract"]
    vectorstore.add_paper(paper_id, title,  None , abstract)

print(len(vectorstore.get_papers()))

{'chunks': 13266, 'vec_chunks': 13266, 'fts_chunks': 13266}
281


In [ ]:
ks = (1,3, 5, 10, 20)

dataset = ingestor.load_qasper_validation()

from src.rag.reranker import Reranker  # or wherever you define it

reranker = Reranker()  # create once

# Start small to measure the time: reranking is slow on CPU
records = run_retrieval(dataset, 20, ret, reranker=reranker, k=max(ks))

for method, question_records in records.items():
    m = compute_metrics(question_records, ks)
    print(f"\n{method}  ({m['questions']} questions)")
    for k in ks:
        print(f"  @{k:<3} hit={m[f'hit@{k}']:.3f}  MRR={m[f'MRR@{k}']:.3f}  "
              f"recall={m[f'recall@{k}']:.3f}  precision={m[f'precision@{k}']:.3f}")

In [ ]:
from src.rag.reranker import Reranker

In [ ]:
# testing the generator on 1 question

from src.rag.generator import Generator

generator = Generator()


In [ ]:

question = "What language pairs are explored?"
paper_id = "1912.01214"

chunks = ret.search(question, k=5, paper_id=paper_id)

answer, stats = generator.generate(question, chunks)

print("Answer:", answer)
print(stats)

# print("Sections used:", [c["section"] for c in chunks])

In [ ]:
print(generator.model)


In [ ]:
import time, httpx

start = time.perf_counter()
r = httpx.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "qwen3:4b-instruct-2507-q4_K_M",
        "messages": [{"role": "user", "content": "Say hello in one sentence."}],
        "stream": False,
        "keep_alive": "30m",
        "options": {"num_ctx": 4096, "num_predict": 50},
    },
    timeout=None,
)
data = r.json()
print(f"wall time: {time.perf_counter() - start:.1f}s", data["message"]["content"])
print("load     :", data["load_duration"] / 1e9, "s")
print("prefill  :", data["prompt_eval_count"], "tokens in", data["prompt_eval_duration"] / 1e9, "s")
print("decode   :", data["eval_count"], "tokens in", data["eval_duration"] / 1e9, "s")
print("decode speed:", data["eval_count"] / (data["eval_duration"] / 1e9), "tok/s")